# 02 · Talking to a local LLM, and RAG

This notebook mirrors the **LLM Chat**, **LLM Playground** and **RAG** pages.
It talks to a local model through **Ollama** (OpenAI-compatible), and shows how
*retrieval-augmented generation* grounds answers in your own documents.

> Start Ollama first (`ollama serve`) and pull a small model, e.g.
> `ollama pull qwen2.5-coder:1.5b`. If Ollama is not running, the chat cells
> fall back to an offline message.

## 1. Connect

In [ ]:
import os
import sys
from pathlib import Path

sys.path.insert(0, str(Path("..").resolve()))      # import the repo's helpers
os.environ.setdefault("LLM_BASE_URL", "http://localhost:11434/v1")
os.environ.setdefault("LLM_MODEL", "qwen2.5-coder:1.5b")

from llm_client import LLMClient, list_ollama_models, ollama_chat

print("installed models:", [m["name"] for m in list_ollama_models()])

## 2. Ask a question (OpenAI-compatible client)

The same `LLMClient` the app uses; it also works with OpenAI by setting `LLM_API_KEY`.

In [ ]:
llm = LLMClient()
print("live endpoint:", llm.is_live)
print(llm.ask("In one sentence, what is machine learning?"))

## 3. Generation settings: how temperature changes the answer

Run the same prompt twice with different `temperature` values.
`0` is focused and repeatable; higher is more random.

In [ ]:
prompt = "Give me a one-line idea for a Python practice exercise."

for temp in (0.0, 1.2):
    out = ollama_chat(
        llm.model,
        [{"role": "user", "content": prompt}],
        base_url=llm.base_url,
        options={"temperature": temp, "num_predict": 60},
    )
    print(f"\ntemperature={temp}  ({out['tokens_per_second']:.1f} tok/s)")
    print(out["content"].strip())

## 4. RAG: answer from *your* documents

The model only knows what is in the text we give it. Here we retrieve the most
relevant lines from a Trivy scan report, then ask the model to answer **using only
that context**.

In [ ]:
from rag_engine import RAGEngine, parse_trivy

text = (Path("..") / "data" / "trivy_scan.txt").read_text(encoding="utf-8")

engine = RAGEngine(split_on="line")          # one chunk per report line
engine.add_text(text, source="trivy_scan.txt")
engine.build()

for chunk in engine.retrieve("which critical vulnerability should I fix?", top_k=3):
    print(f"{chunk.score:.3f}  {chunk.text}")

In [ ]:
query = "Which critical CVE should I patch first, and to what version?"
context = engine.build_context(query, top_k=3)
print("--- context sent to the model ---")
print(context)
print("\n--- grounded answer ---")
print(llm.rag_answer(query, context))

### Bonus: structured parsing

`parse_trivy` turns the report into a table of vulnerabilities.

In [ ]:
import pandas as pd

vulns = parse_trivy(text)
pd.DataFrame([vars(v) for v in vulns])

## Your turn 🏋️

1. Add your own note to the RAG engine (`engine.add_text(...)`) and query it.
2. Try `top_k=1` vs `top_k=5` — how does the retrieved context change?
3. Ask a question the scan **cannot** answer. Does the model admit it, or hallucinate?
4. Swap in a different model with `llm = LLMClient(model="qwen2.5:7b")` and compare speed/quality.
5. (Challenge) Upload a real `trivy image -f json` report, convert it with
   `trivy_json_to_text`, and index that instead.